# Session 9 · KNN: Your First Classifier

**Machine Learning Foundations · Sanketana School of Code**

For four sessions our model answered *"how much?"* — a score, a price, a number on a line. Today the question changes to *"which one?"* — **did the student pass, yes or no?** The answer is a **category**, not a magnitude. That is **classification**.

Same 420 students as Module 2, same habits — we just swap the label from `test_score` (a number) to `passed` (1 or 0). Our first classifier is the most intuitive one there is: **k-Nearest Neighbours**. To guess a new student, it finds the most *similar* past students and lets them **vote**.

By the end of this notebook you will be able to:

- explain KNN in one sentence — *find the k nearest examples and take a majority vote*
- read a **decision boundary**: the map of where the model flips its answer
- turn the **k** knob and predict how the boundary changes
- score a classifier with **accuracy** on a held-out test set (numbers, not vibes)

## Warm-up · Last session's homework

Your coach will walk through Session 8's regression metrics (about 10 minutes) — everyone reported **RMSE, MAE and R²** on the housing model.

That **"numbers, not vibes"** rule doesn't retire today — it follows us into a brand-new kind of model. Every classifier we build gets scored with a named metric too.

## Step 1 · The neighbour idea (no maths)

Here's how *you* would guess whether a new student passes: look at the students most **like** them, and see how those turned out. If the five people closest to you in study hours and attendance all passed, you'd bet you pass too.

That instinct **is** the algorithm:

> To classify a new point, find the **k** training examples closest to it, and let them **vote**. Majority wins.

There's nothing to "fit" — KNN just remembers the training students and, when a new one arrives, measures who is nearest and counts votes. `k` is how many neighbours get a say.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
print("students:", len(students))
print("pass rate:", round(students["passed"].mean(), 3), " (so 'always guess pass' scores about this)")
students.head()

## Step 2 · Scale first — KNN measures distance

"Closest" means **distance**, and distance is only fair if the features share a scale. Look at the raw ranges: attendance runs across ~45 percentage points while practice sessions span 0–7. On raw numbers, attendance would **drown out** every other habit.

So — exactly as in **Session 3** — we scale every feature to the same "one standard step" footing before measuring distance. (KNN works by distance, so scaling is always the right default here.)

In [ ]:
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]

# The raw ranges are wildly different — that is why distance needs scaling:
print("raw feature ranges (max - min):")
print((students[habits].max() - students[habits].min()).round(1))

In [ ]:
X = students[habits].values
y = students["passed"].values

# ✏️ TODO: hide 25% as a test set (stratify keeps the pass/fail mix even).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Scale using the TRAINING set only, then apply to both.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# ✏️ TODO: fit KNN with k=5 and score ACCURACY on the held-out test set.
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print("test accuracy (k=5):", round(knn.score(X_test_s, y_test), 3))
print("You trained a classifier — and judged it with a named metric.")

## Step 3 · See the decision boundary

Because KNN is pure geometry, we can **see** it. We drop to just **two** habits — study hours and attendance — so every student is a dot on a 2-D plot. Then we colour *every point on the map* by what KNN would predict there. The border between the colours is the **decision boundary**: the line where the model changes its mind.

The helper below does the drawing (grids and `contourf` are fiddly — it's written for you). You'll drive it in the next step.

In [ ]:
# Two features so the whole model fits on a 2-D picture.
two = ["study_hours_per_week", "attendance_pct"]
X2 = students[two].values
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y)
scaler2 = StandardScaler().fit(X2_train)
X2_train_s = scaler2.transform(X2_train)
X2_test_s = scaler2.transform(X2_test)

def plot_boundary(k, ax):
    """Fit KNN on the two scaled habits and paint its decision regions."""
    m = KNeighborsClassifier(n_neighbors=k).fit(X2_train_s, y2_train)
    # a grid across the real (unscaled) habit ranges, for readable axes
    x0 = np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 300)
    x1 = np.linspace(X2[:, 1].min() - 2, X2[:, 1].max() + 2, 300)
    gx, gy = np.meshgrid(x0, x1)
    grid = np.c_[gx.ravel(), gy.ravel()]
    zz = m.predict(scaler2.transform(grid)).reshape(gx.shape)
    ax.contourf(gx, gy, zz, alpha=0.25, levels=[-0.5, 0.5, 1.5], cmap="coolwarm")
    ax.scatter(X2[:, 0], X2[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=16)
    ax.set_xlabel("study hours per week")
    ax.set_ylabel("attendance %")
    tr = m.score(X2_train_s, y2_train)
    te = m.score(X2_test_s, y2_test)
    ax.set_title(f"k={k}   train {tr:.2f} / test {te:.2f}")
    return tr, te

fig, ax = plt.subplots(figsize=(6, 5))
plot_boundary(5, ax)
ax.set_title("Decision boundary, k=5  —  blue = pass, red = fail\n" + ax.get_title())
plt.tight_layout(); plt.show()

**Read the picture.** Blue regions are where the model predicts *pass*, red where it predicts *fail*, and the students are the dots. A classifier is really just a way of **carving up the space** into regions — and the boundary is where the answer flips.

## Step 4 · Turn the k knob

KNN has a single dial: **k**, the number of neighbours who vote. Before you run the next cell, **predict**: which will look jagged and which will look smooth?

- **k = 1** — copy your single nearest neighbour.
- **k = 15** — poll a whole neighbourhood and take the majority.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tr1, te1 = plot_boundary(1, axes[0])
tr15, te15 = plot_boundary(15, axes[1])
plt.tight_layout(); plt.show()

print(f"k=1 : train {tr1:.2f} / test {te1:.2f}")
print(f"k=15: train {tr15:.2f} / test {te15:.2f}")

**What you should see.** At **k=1** the boundary is jagged — it wraps tightly around every point, stranding little islands of one colour inside the other — and training accuracy is a perfect **1.00**. At **k=15** the boundary is smooth and one stray point can't flip the vote.

⚠️ That perfect **1.00 on training** at k=1 is a **warning light, not a trophy**: every point's nearest neighbour is *itself*, so k=1 just recites answers it has already seen — it **memorised**. Notice its **test** score is lower. This is the memorising-vs-learning trap from Session 1, and we give it the full treatment in **Session 16**. For now, just train your eye: *perfect on training, worse on test.*

## Step 5 · Choose k with numbers, not vibes

So which k do we ship? Not the prettiest, and definitely not the one with perfect *training* accuracy. We try several values of k, score each on the **held-out test set**, and pick the best there.

In [ ]:
ks = [1, 3, 5, 7, 11, 15, 21, 31]
train_acc, test_acc = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)   # all 5 habits
    train_acc.append(m.score(X_train_s, y_train))
    test_acc.append(m.score(X_test_s, y_test))

plt.figure(figsize=(7, 4))
plt.plot(ks, train_acc, "o-", label="training accuracy")
plt.plot(ks, test_acc, "s-", label="test accuracy")
plt.xlabel("k (number of neighbours voting)")
plt.ylabel("accuracy")
plt.title("Choosing k: training flatters, test is honest")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for k, ta in zip(ks, test_acc):
    print(f"k={k:2d}: test accuracy {ta:.3f}")

Notice **training accuracy is highest at k=1** (it's 1.00 — the memorising trap) and *drops* as k grows, while **test accuracy** starts lower at k=1, climbs, and settles. The k you'd ship is one where the **test** line is high and steady — not the one that aced the training set.

### ✏️ Your turn

Looking at the test-accuracy line: **which k would you ship, and what's the test number that justifies it?** (One or two sentences.)

*Your answer here:*

## Wrap-up

- **Classification** predicts a *category* (pass/fail), not a number.
- **KNN** = find the k nearest examples and take a **majority vote**. Nothing is "fitted" but the stored data itself.
- A **decision boundary** is the model's answer painted across the whole space; the border is where it flips.
- **k** trades memorising for generalising: small k → jagged (k=1 memorises, 100% on train — a warning), large k → smooth.
- **Scale first** (KNN measures distance) and **choose k by test accuracy** — numbers, not vibes.

**Next session:** KNN votes a hard yes/no, but it never says *how confident* it is. Session 10 introduces logistic regression, which returns a **probability** — and shows why a straight line fails for a yes/no question.

*Homework: build a KNN spam filter in `homework.ipynb`.*